# Benchmark k-NN po redukcji wymiarowości do 3D (PCA)

Ten notatnik powtarza **dokładnie ten sam** benchmark k-NN co w sprincie 1 (sanity plots +
skan czasu wykonania dla wszystkich metod i rozmiarów danych), z jedną różnicą: przed
uruchomieniem każdej metody cechy są najpierw rzutowane z oryginalnej przestrzeni (5D) do
**3 wymiarów za pomocą PCA** (liniowa redukcja wymiarowości, w przeciwieństwie do
nieliniowego UMAP i w przeciwieństwie do zero-paddingu ze sprintu 2 -- tu wymiarów faktycznie
ubywa, a nie przybywa, więc część informacji zostaje utracona).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import numpy as np

notebook_dir = Path().resolve()
project_root = notebook_dir.parent
src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.append(src_path)

print(f"Katalog roboczy (notatnik): {notebook_dir}")
print(f"Katalog główny projektu: {project_root}")
print(f"Załadowano ścieżkę modułów: {src_path}")

In [ ]:
import pandas as pd
from sklearn.decomposition import PCA

from hep_tracking.dataset import TrackDataset
from hep_tracking.config import KNNModelConfig
from hep_tracking.models import ScipyCKDTree, SklearnKNN
from hep_tracking.benchmark import ANNBenchmarkRunner
from hep_tracking.plots import plot_3d_hits, plot_distance_distributions, plot_scaling

print("Zależności modułu hep_tracking załadowane pomyślnie!")

def reduce_dataset_to_3d_pca(dataset: TrackDataset, n_components: int = 3, random_state: int = 42) -> tuple[TrackDataset, float]:
    """Redukuje macierz cech X do n_components wymiarów za pomocą PCA.

    Zwraca nowy obiekt TrackDataset ze zredukowanymi cechami oraz udział
    wyjaśnionej wariancji.
    """
    pca = PCA(n_components=n_components, random_state=random_state)
    X_reduced = pca.fit_transform(dataset.X).astype(np.float32)
    explained = pca.explained_variance_ratio_.sum()
    
    reduced_dataset = TrackDataset(X=X_reduced, y=dataset.y, event_ids=dataset.event_ids)
    return reduced_dataset, explained

## Wykresy diagnostyczne (sanity plots) na danych po PCA

Zanim uruchomimy pełny benchmark, sprawdzamy wizualnie, jak wygląda zbiór kontrolny
(`dataset_hard_10k`) po zrzutowaniu do 3D przez PCA -- czy struktura torów jest nadal
widoczna, oraz ile wariancji oryginalnych danych zachowały 3 główne składowe.

In [ ]:
%matplotlib inline

data_dir = project_root / "data"
sanity_dataset_path = data_dir / "dataset_hard_10k.npz"

if sanity_dataset_path.exists():
    dataset_sanity_full = TrackDataset.from_npz(sanity_dataset_path)

    print(f"Wczytano zbiór kontrolny: {sanity_dataset_path.name}")
    print(f"Kształt oryginalnej macierzy cech: {dataset_sanity_full.X.shape}")

    dataset_sanity_pca, explained_sanity = reduce_dataset_to_3d_pca(dataset_sanity_full)
    print(f"Kształt macierzy cech po PCA: {dataset_sanity_pca.X.shape}")
    print(f"Wyjaśniona wariancja (3 składowe): {explained_sanity:.2%}")

    plot_3d_hits(dataset_sanity_pca)
    plot_distance_distributions(dataset_sanity_pca)
else:
    print(f"Nie znaleziono pliku {sanity_dataset_path.name}. Upewnij się, że wygenerowałeś dane.")

**Wnioski z wykresów diagnostycznych (PCA 3D):**

- Jeśli wyjaśniona wariancja jest bliska 100%, oznacza to, że oryginalne 5 wymiarów było w
  dużej mierze redundantne (np. silnie skorelowane ze sobą) i redukcja do 3D nie traci
  prawie żadnej informacji -- tory powinny wyglądać tak samo wyraźnie jak w oryginalnej
  przestrzeni.
- Jeśli wyjaśniona wariancja jest wyraźnie poniżej 100%, PCA odrzuciło część rzeczywistej
  zmienności danych (np. informację o czasie/warstwie, jeśli nie była silnie skorelowana z
  pozycją) -- wtedy należy się spodziewać, że histogram odległości Same Track / Cross Track
  będzie miał **większe nakładanie się** niż w oryginalnej przestrzeni, bo część sygnału
  odróżniającego pary zniknęła wraz z odrzuconymi składowymi.
- W przeciwieństwie do zero-paddingu (sprint 2, 5D→8D, bezstratny) PCA 5D→3D jest z
  definicji **stratne** (tracimy wymiary, a nie dodajemy neutralnych zer) -- to fundamentalna
  różnica, o której trzeba pamiętać przy interpretacji wyników poniżej.

## Benchmark k-NN na danych zredukowanych do 3D (PCA)

Konfiguracja identyczna jak w oryginalnym benchmarku sprintu 1: te same rozmiary danych
$N \in \{1\text{k}, 10\text{k}, 100\text{k}, 1\text{M}\}$, oba tryby (Easy/Hard), $k=5$ i te
same pięć implementacji k-NN. Jedyna zmiana: każdy załadowany zbiór jest najpierw redukowany
do 3D przez PCA, zanim trafi do którejkolwiek z metod.

In [ ]:
target_sizes = {"1k": 1000, "10k": 10000, "100k": 100000, "1M": 1000000}
dataset_modes = ["easy", "hard"]
k_neighbors = 5

models_configs = [
    KNNModelConfig("Scipy cKDTree", ScipyCKDTree, {"workers": -1}),
    KNNModelConfig("Sklearn KDTree", SklearnKNN, {"algorithm": "kd_tree"}),
    KNNModelConfig("Sklearn BallTree", SklearnKNN, {"algorithm": "ball_tree"})
]

runner = ANNBenchmarkRunner(k_neighbors=k_neighbors, warmup_runs=1, num_runs=3)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

explained_variance_log = {}
all_results = []

for mode in dataset_modes:
    print(f"\n{'='*45}")
    print(f" TRYB DANYCH: {mode.upper()}")
    print(f"{'='*45}")

    for size_label, size_val in target_sizes.items():
        filename = data_dir / f"dataset_{mode}_{size_label}.npz"

        if not filename.exists():
            print(f"\n[BRAK PLIKU] {filename.name} - Pomijam.")
            continue

        dataset_full = TrackDataset.from_npz(filename)
        print(f"\n Załadowano: {size_label} ({len(dataset_full)} hitów, {dataset_full.X.shape[1]}D oryginalnie)")

        dataset_pca, explained = reduce_dataset_to_3d_pca(dataset_full)
        explained_variance_log[(mode, size_label)] = explained
        print(f"  > PCA: 5D -> 3D (wyjaśniona wariancja: {explained:.2%})")

        dataset_name = f"{mode}_{size_label}"
        
        exact_knn = ScipyCKDTree(workers=-1)
        exact_knn.fit(dataset_pca.X)
        _, true_idx = exact_knn.kneighbors(dataset_pca.X, k=k_neighbors)

        active_configs = models_configs
        if size_val > 100000:
            active_configs = [c for c in models_configs if c.name != "Sklearn BallTree"]
            print("  > Pomijam: Sklearn BallTree (Zbyt wolne dla 1M)")

        df_size = runner.run(
            models_configs=active_configs, 
            datasets={dataset_name: dataset_pca}, 
            ground_truth={dataset_name: true_idx}
        )
        
        df_size['Mode'] = mode
        df_size['Size_Label'] = size_label
        df_size['Size'] = size_val
        all_results.append(df_size)

print("\nBenchmark (PCA 3D) zakończony sukcesem!")
results_df = pd.concat(all_results, ignore_index=True)
display(results_df.head())

In [ ]:
for mode in dataset_modes:
    df_mode = results_df[results_df['Mode'] == mode]
    
    plot_scaling(
        df=df_mode,
        x_col="Size",
        y_col="Query_Time_s",
        title=f"Skalowalność kNN (PCA 3D) - tryb: {mode.upper()}",
        log_x=True,
        log_y=True
    )

## Wnioski

- **Wpływ na czas wykonania:** przy mniejszej liczbie wymiarów (3D zamiast 5D) każda metoda
  liczy mniej operacji na parę punktów (mniej różnic/mnożeń przy obliczaniu odległości), więc
  spodziewamy się, że wszystkie metody będą co najmniej nieznacznie szybsze niż w oryginalnym
  benchmarku z 5D -- najbardziej powinny to odczuć metody brute-force (koszt liczenia
  odległości skaluje się liniowo z liczbą wymiarów), a najmniej `cKDTree`, który już w 5D był
  bardzo szybki i zdominowany raczej przez $\log N$ niż przez wymiarowość.
- **Ranking metod powinien pozostać ten sam co w 5D** -- redukcja wymiarowości nie zmienia
  charakteru skalowania względem $N$ (brute-force nadal $O(N)$, drzewa nadal $\sim O(\log N)$
  w niskiej wymiarowości) -- zmienia się tylko stała multiplikatywna, nie kształt krzywej.
- **Uwaga na koszt informacyjny:** w przeciwieństwie do eksperymentu z UMAP czy do
  zero-paddingu w sprincie 2, PCA do 3D jest \emph{stratne z definicji}, jeśli wyjaśniona
  wariancja (wypisywana powyżej dla każdego zbioru) jest wyraźnie poniżej 100%. Szybszy
  benchmark k-NN nic nie znaczy, jeśli sąsiedzi znalezieni w zredukowanej przestrzeni
  przestają odpowiadać prawdziwym sąsiadom sprzed redukcji -- to należałoby dodatkowo
  zweryfikować przez recall względem oryginalnego (5D) ground truth, analogicznie do tego,
  jak to zrobiono dla metod ANN w sprincie 2.
- **Praktyczny wniosek:** ten eksperyment mówi nam głównie o **koszcie obliczeniowym**
  redukcji wymiarowości na k-NN, a nie o jej **jakości** -- do pełnej oceny, czy PCA 3D
  nadaje się do dalszych etapów pipeline'u (np. klasyfikacji par), potrzebny byłby dodatkowy
  pomiar recall analogiczny do tego z sanity plots wyżej.